In [1]:
%pip install rank-bm25 sentence-transformers langchain langchain-groq numpy --quiet

In [2]:
import os
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import getpass

# 🔑 Enter Groq API key
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

# ✅ USE THIS MODEL (CURRENTLY WORKING)
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

print("Setup complete")

Enter your Groq API key: ··········
Setup complete


**Corpus (Part 1)**

In [3]:
corpus = [
    "Transformers use self-attention to encode relationships between words in a sequence.",
    "Attention mechanisms compute weighted sums of values based on query-key similarity.",
    "BERT is a bidirectional encoder trained using masked language modeling.",
    "Gradient descent is used to optimize model parameters by minimizing loss.",
    "Stochastic gradient descent updates weights using small random batches of data.",
    "Backpropagation computes gradients to update neural network weights.",
    "The Adam optimizer combines momentum and adaptive learning rates.",
    "BM25 ranks documents using term frequency and inverse document frequency.",
    "Large language models are trained on massive datasets for general-purpose tasks.",
    "Quantization reduces model size by lowering numerical precision."
]

print(f"Corpus size: {len(corpus)}")

Corpus size: 10


**HybridRetriever (Part 2)**

In [4]:
class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k

        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)

        self.sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        doc_vecs = self.sbert.encode(corpus, convert_to_numpy=True)
        self.doc_vecs = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)

    def retrieve(self, query, top_k=5):
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_ranks = {int(d): r+1 for r, d in enumerate(bm25_ranked)}

        q_vec = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_vec = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks = {int(d): r+1 for r, d in enumerate(sbert_ranked)}

        results = []
        for d in range(len(self.corpus)):
            rrf = 1/(self.k+bm25_ranks[d]) + 1/(self.k+sbert_ranks[d])
            results.append({
                "doc_id": d,
                "rrf_score": rrf,
                "bm25_rank": bm25_ranks[d],
                "sbert_rank": sbert_ranks[d],
                "text": self.corpus[d]
            })

        results = sorted(results, key=lambda x: x["rrf_score"], reverse=True)[:top_k]
        return results

hybrid = HybridRetriever(corpus)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Cross Encoder (Part 3)**

In [5]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, top_k=3):
    docs = [c["text"] for c in candidates]
    pairs = [[query, doc] for doc in docs]
    scores = cross_encoder.predict(pairs)

    ranked = np.argsort(scores)[::-1][:top_k]

    return [
        {"text": docs[i], "score": scores[i]}
        for i in ranked
    ]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Query Expansion (HyDE) (Part 4)**

In [6]:
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate a factual paragraph answering the query."),
    ("human", "{query}")
])

hyde_chain = hyde_prompt | llm | StrOutputParser()

def expand_query(query):
    return hyde_chain.invoke({"query": query})

**Advanced RAG Pipeline (Part 5)**

In [10]:
def advanced_rag(user_query):
    expanded = expand_query(user_query)

    candidates = hybrid.retrieve(expanded, top_k=5)

    top_docs = rerank(user_query, candidates, top_k=3)

    # ✅ ADD THIS LINE
    print("Top Docs After Re-ranking:", top_docs)

    context = "\n".join([d["text"] for d in top_docs])

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer using only the context."),
        ("human", "Context:\n{context}\n\nQuestion: {question}")
    ])

    chain = prompt | llm | StrOutputParser()

    return chain.invoke({"context": context, "question": user_query})

**Naïve RAG (Part 6)**

In [11]:
sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vecs = sbert.encode(corpus, convert_to_numpy=True)
doc_vecs = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)

def naive_rag(query):
    q_vec = sbert.encode([query], convert_to_numpy=True)[0]
    q_vec = q_vec / np.linalg.norm(q_vec)
    scores = doc_vecs @ q_vec
    top = np.argmax(scores)
    return corpus[top]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Run comparison (Part 6)**

In [12]:
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is backpropagation?"
]

for q in queries:
    naive = naive_rag(q)
    adv = advanced_rag(q)

    print(f"\nQuery: {q}")
    print(f"Naive Top Doc: {naive}")
    print(f"Advanced Answer: {adv}")

Top Docs After Re-ranking: [{'text': 'Transformers use self-attention to encode relationships between words in a sequence.', 'score': np.float32(5.219939)}, {'text': 'Gradient descent is used to optimize model parameters by minimizing loss.', 'score': np.float32(-11.27047)}, {'text': 'Backpropagation computes gradients to update neural network weights.', 'score': np.float32(-11.286499)}]

Query: how do transformers encode meaning?
Naive Top Doc: Transformers use self-attention to encode relationships between words in a sequence.
Advanced Answer: Transformers use self-attention to encode relationships between words in a sequence. This allows them to capture the context and meaning of words based on their relationships with other words in the sequence.
Top Docs After Re-ranking: [{'text': 'The Adam optimizer combines momentum and adaptive learning rates.', 'score': np.float32(-6.268841)}, {'text': 'Gradient descent is used to optimize model parameters by minimizing loss.', 'score': np.fl

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Different? |
|------|------------------|---------------------|-----------|
| how do transformers encode meaning? | Transformers use self-attention to encode relationships between words in a sequence. | Transformers use self-attention to encode relationships between words in a sequence. | No |
| optimization techniques for training | Gradient descent is used to optimize model parameters by minimizing loss. | The Adam optimizer combines momentum and adaptive learning rates. | Yes |
| what is backpropagation? | Backpropagation computes gradients to update neural network weights. | Backpropagation computes gradients to update neural network weights. | No |